# Comparaison des modèles et optimisation du F1

Objectif : améliorer le premier résultat de `003_modeles.ipynb` en comparant plusieurs modèles et encodages. Aucun score maximal ne peut être garanti.

| Modèle | Encodage et préparation |
| --- | --- |
| Random Forest | Dictionnaires numériques et navigateur en one-hot |
| Extra Trees | Même encodage, arbres davantage randomisés |
| SVM linéaire | Comptages et navigateur, puis pondération TF-IDF |
| Régression logistique | Même représentation TF-IDF |
| Complement Naive Bayes | Même représentation TF-IDF non négative |

## Protocole

| Élément | Choix |
| --- | --- |
| Données | Train uniquement ; caractéristiques reconstruites depuis `002` |
| Séparation | Même 80/20 stratifié, graine 42, que dans `003` |
| Paramètres | Recherche par validation croisée à trois plis sur les 80 % d’apprentissage |
| Critère de sélection | F1 macro, donnant le même poids à chaque utilisateur |
| Évaluation | Scores de validation affichés après la sélection par validation croisée |

Le F1 macro est un choix provisoire : la moyenne utilisée par Kaggle reste à confirmer. La validation 20 % a déjà été examinée dans `003` ; elle n’est donc pas un test final indépendant. Le test Kaggle n’est pas utilisé.

In [1]:
from pathlib import Path
from time import perf_counter
import json
import warnings

import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from threadpoolctl import threadpool_limits

from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, ParameterGrid
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import accuracy_score, f1_score, make_scorer, classification_report
from sklearn.exceptions import ConvergenceWarning

PROJECT_PATH = Path(r"C:\Users\Rog strix Scar\Desktop\BE_MachineLearning")
FEATURES_NOTEBOOK = PROJECT_PATH / "datamanip" / "002_features.ipynb"
RANDOM_STATE = 42
N_JOBS_ARBRES = 2

print("Version scikit-learn :", sklearn.__version__)
print("Notebook de caractéristiques trouvé :", FEATURES_NOTEBOOK.is_file())

Version scikit-learn : 1.8.0
Notebook de caractéristiques trouvé : True


## Reconstruction des caractéristiques

Les cellules de `002_features.ipynb` sont exécutées sans modifier ce fichier. Les dictionnaires contiennent les caractéristiques générales, les comportements et le navigateur.

In [2]:
with FEATURES_NOTEBOOK.open(encoding="utf-8") as fichier:
    notebook_features = json.load(fichier)

espace_features = {"__name__": "__main__"}
for numero, cellule in enumerate(notebook_features["cells"]):
    if cellule["cell_type"] == "code":
        exec(compile("".join(cellule["source"]),
                     f"002_features.ipynb — cellule {numero}", "exec"),
             espace_features)

X_sessions = espace_features["X"].tolist()
y_sessions = espace_features["y"].copy()
df_raw = espace_features["df_raw"]

assert len(X_sessions) == len(y_sessions)
assert all("util" not in ligne and "session_id" not in ligne for ligne in X_sessions)
assert all("navigateur" in ligne for ligne in X_sessions)

print("Sessions :", len(X_sessions), "— Utilisateurs :", y_sessions.nunique())

Fichier trouvé : True
Nombre de sessions : 3279
Nombre d’utilisateurs : 247


,session_id,util,navigateur,traces
0,1,nuh,Firefox,[Création d'un écran(infologic.core.accueil.Ac...
1,2,muz,Google Chrome,[Création d'un écran(infologic.core.gui.contro...
2,3,zrx,Microsoft Edge,[Affichage d'une dialogue(infologic.core.gui.c...
3,4,pou,Firefox,[Création d'un écran(infologic.core.gui.contro...
4,5,ald,Google Chrome,[Affichage d'une dialogue(infologic.acti.modul...


,util,navigateur,types_actions,ecrans_contextes,configurations,chaines
0,nuh,Firefox,"[Création d'un écran, Affichage d'une dialogue...","[infologic.core.accueil.AccueilController, inf...","[DEF_03/24, ACCUEIL_GMAO, 001, ACCUEIL_GMAO, D...","[JCP, JCP, JCP, JCP, JCP, JCP, JCP, JCP, JCP, ..."
1,muz,Google Chrome,"[Création d'un écran, Création d'un écran, Sél...",[infologic.core.gui.controllers.BlankControlle...,"[TELEM_MINIMAL, DEF_03/24, TELEM_MINIMAL, DEF_...","[TEC, TEC, DB, DB, DB, DB, DB, DB, DB, DB, DB,..."
2,zrx,Microsoft Edge,"[Affichage d'une dialogue, Exécution d'un bout...",[infologic.core.gui.controllers.BlankControlle...,"[ACCUEIL_INST, INSTALLATEUR, DEF_07/23, ACCUEI...","[GP, GP, GP, GP, GP, MES, MES, MES, MES, MES, ..."
3,pou,Firefox,"[Création d'un écran, Exécution d'un bouton, A...",[infologic.core.gui.controllers.nested.InputFo...,"[STD, DEF_03/24, STD, DEF_03/24, STD, DEF_03/2...","[JCP, JCP, JCP, JCP, JCP, JCP, JCP, JCP, JCP, ..."
4,ald,Google Chrome,"[Affichage d'une dialogue, Exécution d'un bout...",[infologic.acti.modules.AT_ACTIVITES.compteren...,"[INSTALLATEUR, DEF_03/24, TELEM_MINIMAL, DEF_0...","[ST, ST, ST, GP, GP, GP, GP, GP, GP, GP, GP]"


,util,nb_actions,nb_marqueurs_temporels,dernier_repere_s,sans_actions,sans_marqueur_temporel,nb_actions_modification,part_actions_modification,nb_types_actions_distincts,nb_ecrans_contextes_distincts,nb_configurations_distinctes,nb_chaines_distinctes
0,nuh,2514,614,2905,0,0,4,0.001591,19,7,3,2
1,muz,90,46,230,0,0,26,0.288889,11,5,2,2
2,zrx,608,158,750,0,0,50,0.082237,21,17,7,2
3,pou,886,313,1445,0,0,64,0.072235,19,10,4,1
4,ald,173,60,275,0,0,2,0.011561,14,8,4,2


Session : 1
Utilisateur : nuh


...

Nombre de sessions dans X : 3279
Nombre de réponses dans y : 3279
Nombre de classes : 247


,util,nb_caracteristiques_presentes
0,nuh,62
1,muz,43
2,zrx,80
3,pou,65
4,ald,54


Utilisateur réel : nuh


,valeur
nb_actions,2514
nb_marqueurs_temporels,614
dernier_repere_s,2905
sans_actions,0
sans_marqueur_temporel,0
nb_actions_modification,4
part_actions_modification,0.001591
nb_types_actions_distincts,19
nb_ecrans_contextes_distincts,7
nb_configurations_distinctes,3


Tous les contrôles sont validés.
Sessions : 3279 — Utilisateurs : 247


## Séparation et validation croisée

Le vocabulaire, l’encodage et les pondérations sont réappris dans chaque pli. Les lignes de validation externe ne participent pas au choix des paramètres.

Trois plis sont utilisés car certaines classes ne possèdent que trois exemples dans la partie apprentissage.

In [3]:
X_apprentissage, X_validation, y_apprentissage, y_validation = train_test_split(
    X_sessions, y_sessions,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_sessions,
)

assert set(y_apprentissage.index).isdisjoint(y_validation.index)
assert set(y_apprentissage) == set(y_validation) == set(y_sessions)
assert y_apprentissage.value_counts().min() >= 3

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
classes = sorted(y_sessions.unique())
metriques = {
    "f1_macro": make_scorer(f1_score, average="macro", labels=classes, zero_division=0, pos_label=None),
    "f1_pondere": make_scorer(f1_score, average="weighted", labels=classes, zero_division=0, pos_label=None),
    "accuracy": "accuracy",
}

resume_separation = pd.DataFrame({
    "ensemble": ["Apprentissage", "Validation"],
    "sessions": [len(y_apprentissage), len(y_validation)],
    "utilisateurs": [y_apprentissage.nunique(), y_validation.nunique()],
})
display(resume_separation)

,ensemble,sessions,utilisateurs
0,Apprentissage,2623,247
1,Validation,656,247


## Encodages comparés

| Préparation | Traitement |
| --- | --- |
| `DictVectorizer` | Les comptages restent numériques ; le navigateur devient des colonnes one-hot |
| Arbres | Toutes les caractéristiques de `002`, sans mise à l’échelle |
| TF-IDF | Comptages d’actions et de contextes, plus navigateur ; pas de durées ni de proportions préexistantes |
| Fréquence sous-linéaire | `1 + log(compte)` réduit le poids des répétitions |
| Normalisation L2 | Réduit l’effet de la longueur totale des sessions |

Aucun ordre numérique artificiel n’est imposé aux navigateurs. Les utilisateurs restent des étiquettes textuelles : les classificateurs les acceptent directement.

In [4]:
def garder_comptages(lignes):
    prefixes = (
        "action_nombre::", "ecran_nombre::",
        "configuration_nombre::", "chaine_nombre::",
    )
    return [
        {
            nom: valeur
            for nom, valeur in ligne.items()
            if nom == "navigateur" or nom.startswith(prefixes)
        }
        for ligne in lignes
    ]


def pipeline_arbres(classificateur):
    return Pipeline([
        ("encodage", DictVectorizer(sparse=True, dtype=np.float32)),
        ("classification", classificateur),
    ])


def pipeline_tfidf(classificateur):
    return Pipeline([
        ("comptages", FunctionTransformer(garder_comptages, validate=False)),
        ("encodage", DictVectorizer(sparse=True)),
        ("tfidf", TfidfTransformer(sublinear_tf=True, norm="l2")),
        ("classification", classificateur),
    ])


candidats = {
    "Random Forest": (
        pipeline_arbres(RandomForestClassifier(
            n_estimators=200, class_weight="balanced_subsample",
            random_state=RANDOM_STATE, n_jobs=N_JOBS_ARBRES,
        )),
        {"classification__max_features": ["sqrt", 0.3]},
    ),
    "Extra Trees": (
        pipeline_arbres(ExtraTreesClassifier(
            n_estimators=200, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=N_JOBS_ARBRES,
        )),
        {"classification__max_features": ["sqrt", 0.5]},
    ),
    "SVM TF-IDF": (
        pipeline_tfidf(LinearSVC(
            dual=True, max_iter=10000, random_state=RANDOM_STATE,
        )),
        {
            "classification__C": [1.0, 5.0, 20.0],
            "classification__class_weight": [None, "balanced"],
        },
    ),
    "Régression logistique TF-IDF": (
        pipeline_tfidf(LogisticRegression(
            solver="lbfgs", max_iter=3000, random_state=RANDOM_STATE,
        )),
        {"classification__C": [1.0, 10.0]},
    ),
    "Complement Naive Bayes TF-IDF": (
        pipeline_tfidf(ComplementNB()),
        {"classification__alpha": [0.1, 1.0]},
    ),
}

recherches = {}
resultats_cv = []
for nom, (_, grille) in candidats.items():
    print(nom, ":", len(list(ParameterGrid(grille))), "configurations")

Random Forest : 2 configurations
Extra Trees : 2 configurations
SVM TF-IDF : 6 configurations
Régression logistique TF-IDF : 2 configurations
Complement Naive Bayes TF-IDF : 2 configurations


## Recherche des paramètres

Les cinq familles sont comparées avec les mêmes trois plis et le même F1 macro. Les calculs sont séquentiels pour limiter la mémoire utilisée par les arbres multiclasses.

Les avertissements de convergence sont traités comme des erreurs à résoudre ; aucun échec de modèle n’est masqué.

In [ ]:
for nom, (pipeline, grille) in candidats.items():
    print("Recherche en cours :", nom, flush=True)
    debut = perf_counter()
    recherche = GridSearchCV(
        estimator=pipeline,
        param_grid=grille,
        scoring=metriques,
        refit="f1_macro",
        cv=cv,
        n_jobs=1,
        error_score="raise",
        return_train_score=False,
    )
    with warnings.catch_warnings(), threadpool_limits(limits=2):
        warnings.simplefilter("error", ConvergenceWarning)
        recherche.fit(X_apprentissage, y_apprentissage)

    recherches[nom] = recherche
    indice = recherche.best_index_
    resultats_cv = [ligne for ligne in resultats_cv if ligne["modele"] != nom]
    resultats_cv.append({
        "modele": nom,
        "f1_macro_cv": recherche.best_score_,
        "ecart_type_cv": recherche.cv_results_["std_test_f1_macro"][indice],
        "f1_pondere_cv": recherche.cv_results_["mean_test_f1_pondere"][indice],
        "accuracy_cv": recherche.cv_results_["mean_test_accuracy"][indice],
        "duree_s": round(perf_counter() - debut, 2),
        "parametres": recherche.best_params_,
    })
    print("F1 macro CV :", round(recherche.best_score_, 4), flush=True)

classement_cv = pd.DataFrame(resultats_cv).sort_values(
    "f1_macro_cv", ascending=False
).reset_index(drop=True)

nom_retenu = classement_cv.iloc[0]["modele"]
modele_retenu = recherches[nom_retenu].best_estimator_

display(classement_cv.round(4))
print("Choix effectué par CV uniquement :", nom_retenu)

Recherche en cours : Random Forest


## Évaluation sur la validation réservée

Chaque famille est évaluée avec ses paramètres choisis en validation croisée. Le modèle retenu reste celui choisi par le F1 macro CV, même si l’ordre des scores externes diffère.

Ces scores ne constituent pas une garantie sur le test Kaggle.

In [ ]:
def scores_validation(y_reel, y_predit):
    return {
        "accuracy": accuracy_score(y_reel, y_predit),
        "f1_macro": f1_score(y_reel, y_predit, average="macro", labels=classes, zero_division=0),
        "f1_pondere": f1_score(y_reel, y_predit, average="weighted", labels=classes, zero_division=0),
    }


predictions_modeles = {}
resultats_validation = []
for nom, recherche in recherches.items():
    with threadpool_limits(limits=2):
        predictions = recherche.predict(X_validation)
    predictions_modeles[nom] = predictions
    resultats_validation.append({
        "modele": nom,
        "retenu_par_cv": nom == nom_retenu,
        **scores_validation(y_validation, predictions),
    })

classement_validation = pd.DataFrame(resultats_validation).sort_values(
    "f1_macro", ascending=False
).reset_index(drop=True)

display(classement_validation.round(4))
print("Modèle retenu par CV :", nom_retenu)

,modele,retenu_par_cv,accuracy,f1_macro,f1_pondere
0,Extra Trees,True,0.9162,0.8876,0.9049
1,Random Forest,False,0.9040,0.8654,0.8872
2,SVM TF-IDF,False,0.8872,0.8509,0.8748
3,Régression logistique TF-IDF,False,0.8811,0.8389,0.8649
4,Complement Naive Bayes TF-IDF,False,0.4421,0.2828,0.3557


Modèle retenu par CV : Extra Trees


## Détail des paramètres explorés

Ce tableau permet de comprendre quels réglages ont été meilleurs dans les plis d’apprentissage, sans utiliser la validation réservée.

In [ ]:
details_recherche = []
for nom, recherche in recherches.items():
    tableau = pd.DataFrame(recherche.cv_results_)[[
        "params", "mean_test_f1_macro", "std_test_f1_macro", "rank_test_f1_macro"
    ]].copy()
    tableau.insert(0, "modele", nom)
    details_recherche.append(tableau)

display(pd.concat(details_recherche, ignore_index=True).sort_values(
    ["modele", "rank_test_f1_macro"]
).round(4))

,modele,params,mean_test_f1_macro,std_test_f1_macro,rank_test_f1_macro
13,Complement Naive Bayes TF-IDF,{'classification__alpha': 1.0},0.2816,0.0094,1
12,Complement Naive Bayes TF-IDF,{'classification__alpha': 0.1},0.2729,0.0074,2
3,Extra Trees,{'classification__max_features': 0.5},0.8775,0.0130,1
2,Extra Trees,{'classification__max_features': 'sqrt'},0.8460,0.0104,2
0,Random Forest,{'classification__max_features': 'sqrt'},0.8448,0.0071,1
1,Random Forest,{'classification__max_features': 0.3},0.8204,0.0173,2
11,Régression logistique TF-IDF,{'classification__C': 10.0},0.7940,0.0126,1
10,Régression logistique TF-IDF,{'classification__C': 1.0},0.3556,0.0123,2
9,SVM TF-IDF,"{'classification__C': 20.0, 'classification__c...",0.8519,0.0118,1
7,SVM TF-IDF,"{'classification__C': 5.0, 'classification__cl...",0.8506,0.0117,2


## Erreurs du modèle retenu

Les faibles effectifs de validation rendent certains F1 par utilisateur très instables. Le numéro de session sert uniquement à retrouver les traces d’origine.

In [ ]:
predictions_retenues = predictions_modeles[nom_retenu]
rapport = classification_report(
    y_validation, predictions_retenues,
    labels=classes, output_dict=True, zero_division=0,
)
rapport_utilisateurs = pd.DataFrame(rapport).T.loc[classes].rename(columns={
    "recall": "rappel", "f1-score": "f1", "support": "sessions_validation",
})
rapport_utilisateurs["sessions_apprentissage"] = (
    y_apprentissage.value_counts().reindex(rapport_utilisateurs.index)
)
display(rapport_utilisateurs.sort_values("f1").head(15).round(3))

analyse = pd.DataFrame({
    "session_id": df_raw.loc[y_validation.index, "session_id"],
    "util_reel": y_validation,
    "util_predit": predictions_retenues,
})
erreurs = analyse.loc[analyse["util_reel"] != analyse["util_predit"]].copy()
print("Erreurs du modèle retenu :", len(erreurs), "/", len(y_validation))
display(erreurs.head(15))

,precision,rappel,f1,sessions_validation,sessions_apprentissage
eqw,0.0,0.000,0.0,1.0,5
jew,0.0,0.000,0.0,1.0,3
itt,0.0,0.000,0.0,1.0,3
hll,0.0,0.000,0.0,2.0,7
nkb,0.0,0.000,0.0,1.0,6
tmn,0.0,0.000,0.0,1.0,5
pou,0.0,0.000,0.0,1.0,6
oph,0.0,0.000,0.0,1.0,6
uhx,0.0,0.000,0.0,1.0,4
xgb,0.0,0.000,0.0,1.0,6


Erreurs du modèle retenu : 55 / 656


,session_id,util_reel,util_predit
1667,1668,ajo,szr
2993,2994,sbz,xkr
676,677,zbn,kwi
420,421,oph,hjs
615,616,oec,vyu
488,489,uwp,kks
2501,2502,zct,cjr
258,259,eqw,tvt
1591,1592,zqs,wjn
87,88,kks,kwi


## Contrôles finaux

In [ ]:
assert len(recherches) == len(candidats) == 5
assert len(X_apprentissage) + len(X_validation) == len(X_sessions)
assert set(y_apprentissage.index).isdisjoint(y_validation.index)
assert espace_features["df_raw"].equals(
    espace_features["df_train"][espace_features["df_raw"].columns]
)
assert nom_retenu == classement_cv.iloc[0]["modele"]

for nom, recherche in recherches.items():
    assert recherche.n_splits_ == 3
    assert len(predictions_modeles[nom]) == len(y_validation)
    assert set(predictions_modeles[nom]).issubset(set(y_apprentissage))
    noms = recherche.best_estimator_.named_steps["encodage"].get_feature_names_out()
    assert any(nom_variable.startswith("navigateur=") for nom_variable in noms)

assert classement_validation[["accuracy", "f1_macro", "f1_pondere"]].notna().all().all()
print("Tous les contrôles sont validés.")

Tous les contrôles sont validés.


## Synthèse

Le modèle choisi est celui qui maximise le F1 macro moyen parmi les configurations essayées en validation croisée. Ce n’est pas la preuve d’un maximum absolu.

In [ ]:
synthese = classement_cv[["modele", "f1_macro_cv", "ecart_type_cv"]].merge(
    classement_validation, on="modele"
)
for colonne in ["f1_macro_cv", "ecart_type_cv", "accuracy", "f1_macro", "f1_pondere"]:
    synthese[colonne] = (synthese[colonne] * 100).round(2)

synthese = synthese.rename(columns={
    "f1_macro_cv": "F1 macro CV (%)",
    "ecart_type_cv": "Écart-type CV (points)",
    "accuracy": "Accuracy validation (%)",
    "f1_macro": "F1 macro validation (%)",
    "f1_pondere": "F1 pondéré validation (%)",
})
display(synthese)
print("Modèle retenu :", nom_retenu)
print("Paramètres :", recherches[nom_retenu].best_params_)

,modele,F1 macro CV (%),Écart-type CV (points),retenu_par_cv,Accuracy validation (%),F1 macro validation (%),F1 pondéré validation (%)
0,Extra Trees,87.75,1.30,True,91.62,88.76,90.49
1,SVM TF-IDF,85.19,1.18,False,88.72,85.09,87.48
2,Random Forest,84.48,0.71,False,90.40,86.54,88.72
3,Régression logistique TF-IDF,79.40,1.26,False,88.11,83.89,86.49
4,Complement Naive Bayes TF-IDF,28.16,0.94,False,44.21,28.28,35.57


Modèle retenu : Extra Trees
Paramètres : {'classification__max_features': 0.5}


## Interprétation des résultats obtenus

| Modèle | F1 macro CV | F1 macro validation | Accuracy validation |
| --- | ---: | ---: | ---: |
| Extra Trees | 87,75 % | 88,76 % | 91,62 % |
| SVM TF-IDF | 85,19 % | 85,09 % | 88,72 % |
| Random Forest | 84,48 % | 86,54 % | 90,40 % |
| Régression logistique TF-IDF | 79,40 % | 83,89 % | 88,11 % |
| Complement Naive Bayes TF-IDF | 28,16 % | 28,28 % | 44,21 % |

**Choix : Extra Trees**, sélectionné par la validation croisée sur l’apprentissage, avec 200 arbres, `max_features=0.5` et `class_weight="balanced"`.

Le F1 macro sur la validation passe de **69,72 %** pour le SVM standard de `003` à **88,76 %**, soit environ **19,04 points** supplémentaires sur la même séparation. Les modèles et les prétraitements diffèrent : ce gain ne peut pas être attribué au seul encodage.

Extra Trees obtient aussi le meilleur score externe parmi les familles essayées. La faiblesse de Naive Bayes montre qu’un modèle supplémentaire ou un encodage différent n’améliore pas automatiquement le F1. Ces résultats restent limités aux configurations et aux sessions évaluées.

## Limites et suite

| Point | À retenir |
| --- | --- |
| Comparaison | Cinq familles, paramètres choisis sur trois plis d’apprentissage |
| F1 | Macro utilisé provisoirement ; métrique officielle à confirmer |
| Validation externe | Identique à celle déjà examinée dans `003` ; pas un test final indépendant |
| Optimisation | Meilleur des réglages essayés, sans garantie de maximum absolu |
| Données rares | Peu d’exemples pour certains utilisateurs |
| Test Kaggle | Non lu et aucune soumission créée |

Avant la soumission, confirmer la stabilité des choix si nécessaire, puis réentraîner la chaîne retenue sur toutes les sessions étiquetées. Le vocabulaire et le TF-IDF devront alors être réappris sur tout le train avant de transformer le test.

### Documentation

- [DictVectorizer : encodage des dictionnaires](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.DictVectorizer.html)
- [TF-IDF : pondération des comptages](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html)
- [GridSearchCV : recherche par validation croisée](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)
- [Random Forest](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)